# 02 — Pipeline semanal, búsqueda y visualización
## Caso Tipo C: Sueño en los Recién Nacidos

Este notebook crea una secuencia de registros semanales sintéticos, calcula un costo y recorre los registros disponibles para buscar el primero que cumpla el objetivo. Las curvas son datos inventados para demostrar el proceso.

## 1. Importar bibliotecas y generar el DataFrame

In [ ]:
import heapq
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
n = 12

df = pd.DataFrame({
    "Etiqueta": [f"Semana {i+1}" for i in range(n)],
    "X1_Sueno": np.clip(np.linspace(4.0, 8.0, n) + rng.normal(0, 0.8, n), 2, 10),
    "X2_Despertarse": rng.integers(0, 6, n),
    "X3_Fatiga": rng.integers(0, 5, n),
    "X4_Rutina": rng.integers(0, 3, n),
})
df["X1_Sueno"] = df["X1_Sueno"].round(1)

## 2. Calcular el costo de cada estado

La fórmula coincide con el notebook anterior para mantener consistencia. Es un puntaje de práctica, no una escala clínica.

In [ ]:
def calcular_costo_fila(fila):
    return (
        max(0, (10 - fila["X1_Sueno"]) * 15)
        + fila["X2_Despertarse"] * 20
        + fila["X3_Fatiga"] * 15
        + (2 - fila["X4_Rutina"]) * 10
    )

df["Costo_Total"] = df.apply(calcular_costo_fila, axis=1)

display(df.round(2))

## 3. Representar estados y definir una heurística

Cada fila es un estado. La transición permitida es pasar de la semana actual a la siguiente. La heurística mide la diferencia absoluta entre el costo del estado y el umbral. Esta heurística es ilustrativa; por sí sola no convierte el procedimiento en una optimización clínica.

In [ ]:
class EstadoSueno:
    def __init__(self, paso, costo):
        self.paso = paso
        self.costo = float(costo)

    def __lt__(self, other):
        return self.costo < other.costo

def heuristica(costo_actual, costo_objetivo):
    return abs(costo_objetivo - costo_actual)

## 4. Buscar un estado que cumpla el objetivo

Se recorren transiciones consecutivas y se usa una cola de prioridad. El objetivo es hallar un estado con costo menor o igual al umbral. Si ninguno cumple, el algoritmo devuelve el estado de menor costo entre los registros evaluados. El resultado está limitado a este DataFrame.

In [ ]:
def busqueda_estados_sueno(df, costo_objetivo):
    if df.empty:
        return None, None

    inicio = EstadoSueno(0, df.iloc[0]["Costo_Total"])
    frontera = [(inicio.costo + heuristica(inicio.costo, costo_objetivo), inicio)]
    visitados = set()
    mejor = inicio

    while frontera:
        _, actual = heapq.heappop(frontera)

        if actual.costo < mejor.costo:
            mejor = actual

        if actual.costo <= costo_objetivo:
            return actual.paso, actual.costo

        if actual.paso in visitados:
            continue
        visitados.add(actual.paso)

        siguiente_paso = actual.paso + 1
        if siguiente_paso < len(df) and siguiente_paso not in visitados:
            siguiente = EstadoSueno(
                siguiente_paso,
                df.iloc[siguiente_paso]["Costo_Total"]
            )
            f = siguiente.costo + heuristica(siguiente.costo, costo_objetivo)
            heapq.heappush(frontera, (f, siguiente))

    return mejor.paso, mejor.costo

costo_meta = 55.0
paso_optimo, costo_optimo = busqueda_estados_sueno(df, costo_meta)

if paso_optimo is not None:
    fila = df.iloc[int(paso_optimo)]
    print(f"Estado seleccionado: {fila['Etiqueta']}")
    print(f"Costo de desviación: {costo_optimo:.2f}")
    print(f"X1 (Sueño): {fila['X1_Sueno']:.1f} h")
    print(f"X2 (Despertares): {int(fila['X2_Despertarse'])}")
    print(f"X3 (Fatiga): {int(fila['X3_Fatiga'])}")
    print(f"X4 (Rutina): {int(fila['X4_Rutina'])}")
    if costo_optimo > costo_meta:
        print("No se alcanzó el umbral; se devuelve el estado de menor costo evaluado.")
else:
    print("No hay estados disponibles.")

## 5. Visualización de líneas

In [ ]:
plt.figure(figsize=(12, 6))

plt.plot(df["Etiqueta"], df["X1_Sueno"], label="X1: Sueño continuo (h)", marker="o")
plt.plot(df["Etiqueta"], df["X2_Despertarse"], label="X2: Despertares", marker="s")
plt.plot(df["Etiqueta"], df["X3_Fatiga"], label="X3: Fatiga de padres", marker="^")
plt.plot(df["Etiqueta"], df["X4_Rutina"], label="X4: Rutina (escala ficticia)", marker="d")

if paso_optimo is not None:
    plt.axvline(
        x=paso_optimo,
        linestyle="--",
        linewidth=2,
        label="Estado seleccionado"
    )

plt.title("Evolución sintética de variables del sistema de sueño infantil")
plt.xlabel("Periodo simulado")
plt.ylabel("Valor de la variable (escalas distintas)")
plt.xticks(rotation=45)
plt.legend()
plt.grid(True, linestyle=":", alpha=0.7)
plt.tight_layout()
plt.show()

## 6. Visualizar el costo y el umbral

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(df["Etiqueta"], df["Costo_Total"], marker="o", label="Costo total sintético")
plt.axhline(costo_meta, linestyle="--", label=f"Umbral objetivo = {costo_meta}")
if paso_optimo is not None:
    plt.scatter(
        df.iloc[int(paso_optimo)]["Etiqueta"],
        df.iloc[int(paso_optimo)]["Costo_Total"],
        s=100,
        label="Estado seleccionado"
    )
plt.title("Costo de desviación por periodo simulado")
plt.xlabel("Periodo simulado")
plt.ylabel("Costo (ficticio)")
plt.xticks(rotation=45)
plt.legend()
plt.grid(True, linestyle=":", alpha=0.7)
plt.tight_layout()
plt.show()

## 7. Conclusiones
El notebook construye un DataFrame semanal sintético, calcula un costo unificado, aplica una búsqueda sobre los estados disponibles y presenta gráficos de líneas y costos. Como las transiciones solo conectan cada registro con el siguiente, este ejemplo es una demostración didáctica de búsqueda, no una solución clínica ni una prueba de que se haya optimizado el sueño de un bebé.